# 02. The signal: last month's realised variance, and the position each construction takes from it

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Factor | the monthly or daily return of one long-short portfolio that captures one source of common movement in stock returns: Mkt-RF, SMB, HML, Mom, RMW and CMA, as in notebook 01 |
| Excess return | a return minus the one-month Treasury bill rate over the same period; every factor is one |
| Trading day | a day on which the New York Stock Exchange traded, one row of French's daily files |
| Realised variance | the variance of a factor computed from the daily returns of one month. Last month's realised variance is the signal this study evaluates |
| Position | the multiple of the factor held in a month: a position of 2 doubles every gain and loss of the factor, 0.5 halves them. The papers call a position above 1 leverage |
| Volatility-managed factor | Moreira and Muir's construction: a position of c divided by last month's realised variance, so the position is small after a volatile month and large after a calm one |
| c | the constant of the volatility-managed factor, set so that its standard deviation over the sample equals the original factor's |
| Percentile | the value below which a given share of the months fall: the 95th percentile (P95) is exceeded in 5 months out of 100 |
| Covariance matrix | the table of the variances of two or more return series and the covariances between each pair |
| Mean-variance weights | the amounts of each return series an investor holds who trades off the mean of the portfolio's return against its variance: (1/γ) times the inverse of the covariance matrix times the vector of means |
| γ (gamma) | risk aversion: how much variance the investor gives up for one unit of mean return. A larger γ gives smaller positions |
| Real-time combination | Cederburg, O'Doherty, Wang and Yan's construction: each month, mean-variance weights on the volatility-managed and the original factor, estimated from the months before it only. Its return is S1 |
| Real-time original | the same with the original factor alone. Its return is S2 |
| Training window | the months a real-time estimate is computed from: every month from the factor's first month to the month before the one being invested in |
| Cap | the largest position the real-time constructions may take, 5 in either direction |
| Vintage | the version of French's files on the download date, named by the CRSP release they were built from |
| Provenance | the record of what was downloaded, when, with its checksum and vintage, written to `outputs/provenance_french.json` |

**What this notebook does.** It builds the signal, last month's realised variance, for each of the six factors under the three papers' definitions, and turns it into a position in two ways: Moreira and Muir's volatility-managed factor, on their sample, and Cederburg et al.'s real-time combination, month by month from the end of its first ten-year training window to August 2026. It reports how large the positions are, works one month of each construction by hand, and writes every series to `outputs/`. No return, Sharpe ratio or alpha is computed here.

**Why it comes at this point.** Every later notebook compares returns of positions built here. Before any comparison, the positions themselves have to be right: computed from the months before the one invested in, with each paper's own definition, and of a size a reader can check against the papers' description.

**Which expectations it judges.** None. E6 judges the percentiles of the volatility-managed position against the papers' Table V and Table 1, Panel D, in notebook 08; this notebook reports the 5th, 50th and 95th percentiles and the maximum, which the design names for describing the position.

**The papers.** Moreira and Muir (2017), "Volatility-Managed Portfolios", Journal of Finance 72(4), 1611-1644; Cederburg, O'Doherty, Wang and Yan (2020), "On the performance of volatility-managed portfolios", Journal of Financial Economics 138(1), 95-117; Barroso and Santa-Clara (2015), "Momentum has its moments", Journal of Financial Economics 116(1), 111-120. Page numbers are the journals'.

**The three definitions of the signal.**

- Moreira and Muir, equation 2, p. 1616: each day's return minus the month's mean daily return, squared, summed over the month's trading days. A month with more trading days gives a larger sum at the same daily risk.
- Cederburg et al., equation 4, p. 98: the sum of the squared daily returns, times 22 divided by the month's number of trading days J. Every month is rescaled to 22 trading days.
- Barroso and Santa-Clara, equation 5, p. 115: 21 times the mean squared daily return over the last 126 trading days. The window is fixed in trading days and runs across month ends.

Each comparison with a paper's printed numbers uses that paper's definition (`RV_BY_COMPARISON` in the constants).

**The real-time combination's three numbers, and why each has its value.** All three are Cederburg et al.'s own (Table 5 note and pp. 106-107), so the replication matches their baseline.

- The training window starts at 120 months (`REALTIME_K`), ten years, and grows by one month each month (an expanding window). Ten years leaves most of the sample for investing; Cederburg et al. report that longer starting windows and rolling windows do not change their conclusion (Table 7, p. 110).
- γ = 5 (`REALTIME_GAMMA`) is the risk aversion they use throughout; it scales every position by the same factor, so it changes the size of S1 and S2 and leaves their Sharpe ratios unchanged as long as the cap does not bind.
- The cap of 5 (`REALTIME_LEVERAGE_CAP`), applied by truncating the position at 5 or at -5. Mean-variance weights estimated from a window can be extreme, and the cap keeps the months with extreme estimates from dominating the result (p. 107).

**What I expect to see, and where it comes from.**

- No month below 15 trading days for any factor (notebook 01).
- For the market, the ratio of Cederburg et al.'s realised variance to Moreira and Muir's is close to 22 divided by one less than the month's trading days, 22 / (J - 1). Two things set it. Cederburg et al. rescale the sum of squares to 22 trading days, which gives 22 / J. Moreira and Muir subtract the month's own mean before squaring, and a mean estimated from the same J days absorbs on average one day's worth of variation: with 21 trading days, the sum of squared deviations from the month's mean is on average 20/21 of the sum of squares. Together, 22 / J times J / (J - 1) is 22 / (J - 1): a median near **0.94** before June 1952 (22 / 23.5, with 24.5 trading days on average) and near **1.10** from 1953 (22 / 20.0, with 21.0).
- The volatility-managed position on Moreira and Muir's sample has a median near 1, because c is set so that the managed factor's standard deviation equals the original's, and a long right tail: for every factor the 95th percentile is more than twice the median, as the published percentiles imply (Moreira and Muir's Table V gives the market's 90th percentile as 2.64 against a median of 0.93, p. 1626). The 5th, 50th and 95th percentiles and the maximum are reported for each factor.
- The worked month: on 19 October 1987 the market had its largest one-day fall since 1926. October's realised variance is the largest of the 1980s, so the volatility-managed position for November 1987 is a small fraction of the median position.
- The real-time combination's first position is in the 121st month of each factor: **August 1936** for Mkt-RF, SMB and HML, **January 1937** for Mom and **August 1973** for RMW and CMA. To August 2026 that gives **1,081**, **1,076** and **637** months with a position.
- The real-time position does not depend on c: multiplying the managed factor by any constant divides its mean-variance weight by the same constant, so the product x_sigma times c, and the position, stay the same. The notebook checks it on one month.

## How this notebook is organised

1. The modules are written into the runtime and the factors are loaded.
2. The signal: the three definitions for the six factors, compared for the market, and the months below 15 trading days.
3. A worked month: October 1987, by hand.
4. The volatility-managed factor on Moreira and Muir's sample: c and the size of the position.
5. The real-time combination: one month worked in full, then the positions of every month to August 2026.
6. A figure of the market's signal and both positions.
7. The outputs, and what could be wrong.

In [ ]:
import os
os.makedirs("src/sq", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
open("src/sq/__init__.py", "w").close()

In [ ]:
%%writefile src/sq/constants.py
"""
Pre-committed numbers for the study.

Every sample boundary, construction parameter, expectation and test setting is
defined here, one per name, with its source beside it. A notebook may read
these names; it may not introduce numbers of its own. The design was fixed on
8 November 2026 and restated against the three papers' printed tables on
10 November 2026, before any code that judges an expectation was written.

The three papers, with the journal's page numbers used throughout:

  MM    Moreira and Muir (2017), "Volatility-Managed Portfolios", Journal of
        Finance 72(4), 1611-1644.
  COWY Cederburg, O'Doherty, Wang and Yan (2020), "On the performance of
        volatility-managed portfolios", Journal of Financial Economics 138(1),
        95-117.
  BSC   Barroso and Santa-Clara (2015), "Momentum has its moments", Journal of
        Financial Economics 116(1), 111-120.

The Sharpe ratio test is Ledoit and Wolf (2008), "Robust performance
hypothesis testing with the Sharpe ratio", Journal of Empirical Finance 15(5),
850-859. The published numbers the expectations compare against are typed in
src/sq/published.py, never here.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":       "F-F_Research_Data_Factors_CSV.zip",             # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",       # the same, daily from 1926-07-01
    "factors5":       "F-F_Research_Data_5_Factors_2x3_CSV.zip",       # adds RMW and CMA, monthly from 1963-07
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # the same, daily from 1963-07-01
    "momentum":       "F-F_Momentum_Factor_CSV.zip",                   # Mom, monthly from 1927-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",             # Mom, daily from 1926-11-03
    "mom10":          "10_Portfolios_Prior_12_2_CSV.zip",              # ten portfolios formed on momentum, monthly from 1927-01
    "mom10_daily":    "10_Portfolios_Prior_12_2_Daily_CSV.zip",        # the same, daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# The six factors, by French's column names, and the file each comes from.
# Mkt-RF, SMB and HML come from the three-factor file, as in MM Table I and
# COWY section 2.1 (Fama and French 1993); RMW and CMA from the five-factor
# file (Fama and French 2015); Mom from the momentum file. RF is the
# three-factor file's one-month Treasury bill return.
FACTORS = ("Mkt-RF", "SMB", "HML", "Mom", "RMW", "CMA")
FACTOR_SOURCE = {
    "Mkt-RF": "factors3", "SMB": "factors3", "HML": "factors3",
    "Mom": "momentum", "RMW": "factors5", "CMA": "factors5",
}
RF_SOURCE = "factors3"

# WML, the momentum series of BSC: the top decile minus the bottom decile of
# the ten value-weighted portfolios formed on returns from month t-12 to t-2
# (BSC Appendix A, p. 119). Block 0 of both files holds the value-weighted returns.
WML_LONG = "Hi PRIOR"
WML_SHORT = "Lo PRIOR"
WML_BLOCK = 0
WML_BLOCK_TITLE_MUST_CONTAIN = "Value Weight"

# The end of the sample, fixed on 8 November 2026: August 2026 on any vintage.
# The loader cuts every file at this month, so that each notebook sees the same
# months on any later vintage; the vintage line and the SHA-256 in the
# provenance record say which revision of French's history a run used. A run on
# a later vintage, cut at August 2026, is a confirmation run: it is labelled by
# its vintage in the provenance record, it is expected to reproduce every
# verdict line and every printed decimal and not every digit, because French
# revises history, and it replaces no committed table without a dated decision.
SAMPLE_END = "2026-08"
SAMPLE_END_RULE_DATE = "2026-11-08"
CONFIRMATION_RUN = "a run on a later vintage, cut at August 2026, labelled by its vintage in the provenance record"

# French's daily files carry Saturday trading days until 24 May 1952. The number of
# trading days in a calendar month runs from 17 to 27 before June 1952 and from 15
# to 23 from 1953 on (counted on vintage 202609). Months with fewer trading days
# than the level below are flagged and reported; none is dropped.
LAST_SATURDAY_TRADING_DAY = "1952-05-24"
TRADING_DAYS_FLAG_BELOW = 15

PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 2. The samples
# ---------------------------------------------------------------------------

# The first month with a volatility-managed return, for each factor: the month
# after the first month of daily data, or the first month of the monthly file
# if that is later. These are COWY's start dates (Table 1 note, p. 99).
FACTOR_FIRST_MONTH = {
    "Mkt-RF": "1926-08", "SMB": "1926-08", "HML": "1926-08",
    "Mom": "1927-01", "RMW": "1963-08", "CMA": "1963-08",
}

# MM's sample. The paper says "1926 to 2015" (Table I note, p. 1619) and prints
# N = 1,065 for Mkt, SMB and HML, 1,060 for Mom and 621 for RMW and CMA. From
# the first months above, all three counts end in April 2015 exactly; an end in
# December 2015 would give 1,073, 1,068 and 629. The paper does not state the
# months, so the end is inferred from these counts; the computed N is reported
# beside the published N in every table that judges E1.
MM_SAMPLE_END = "2015-04"
MM_SAMPLE_END_IS_INFERRED = True
MM_EXPECTED_N = {"Mkt-RF": 1065, "SMB": 1065, "HML": 1065, "Mom": 1060, "RMW": 621, "CMA": 621}

# COWY's sample: the same first months to December 2016 (Table 1 note, p. 99).
COWY_SAMPLE_END = "2016-12"
COWY_EXPECTED_N = {"Mkt-RF": 1085, "SMB": 1085, "HML": 1085, "Mom": 1080, "RMW": 641, "CMA": 641}

# The months after MM's sample, the study's own question (E5).
AFTER_START = "2015-05"
AFTER_END = SAMPLE_END
AFTER_EXPECTED_MONTHS = 136

# BSC's sample is 1927:03 to 2011:12 (Table 3, p. 116). Their daily WML starts
# in August 1926 (Daniel and Moskowitz's data, Appendix A, p. 119); French's
# daily file of the ten momentum portfolios starts on 3 November 1926, so the
# 126 trading days the variance needs are first available at the end of April 1927
# and the first scaled month is May 1927, two months after the paper's.
BSC_PAPER_FIRST_MONTH = "1927-03"
BSC_FIRST_MONTH = "1927-05"
BSC_SAMPLE_END = "2011-12"
BSC_EXPECTED_MONTHS = 1016          # May 1927 to December 2011 inclusive
BSC_EXTENDED_END = SAMPLE_END       # the second period of scaled momentum: May 1927 to August 2026

# ---------------------------------------------------------------------------
# 3. The constructions
# ---------------------------------------------------------------------------

# The signal: last month's realised variance of the factor's daily returns.
# Each construction uses its own paper's definition.
#
# MM equation 2, p. 1616: each day's return minus the month's mean, squared and
# summed over the month. The equation is written for a 22-day month; the mean
# here is the month's own mean, the sum of its daily returns divided by its
# number of trading days.
RV_MM = "sum over the month's trading days of (daily return minus the month's mean daily return) squared"
# COWY equation 4, p. 98: 22 divided by the month's number of trading days J,
# times the sum of squared daily returns; no mean is removed.
RV_COWY_DAYS = 22
RV_COWY = "22 / J times the sum over the month's J trading days of the squared daily return"
# BSC equation 5, p. 115: 21 times the mean squared daily return over the last
# 126 trading days up to the month's last trading day, a monthly variance.
RV_BSC_TRADING_DAYS = 126
RV_BSC_DAYS_PER_MONTH = 21

# Which definition each comparison uses: a comparison with a paper's printed
# numbers uses that paper's definition. Fixed 10 November 2026.
RV_BY_COMPARISON = {
    "volatility-managed factor on MM's sample: E1, E4, E6 (Table IV, Table V)": "MM",
    "volatility-managed factor on COWY's sample: E2 (i), E6 (Table 1, Panel D)": "COWY",
    "real-time combination and real-time original: E2 (ii) to (iv), E5": "COWY",
    "volatility-managed factor after the sample, c fixed to April 2015: E5, no verdict": "MM",
    "scaled WML: E3": "BSC",
}

# The volatility-managed factor (MM equation 1, p. 1616): f_sigma_t = (c / RV_{t-1}) f_t,
# with c set so that the managed and the original factor have the same standard
# deviation over the sample the comparison runs on. c does not change a Sharpe
# ratio or an appraisal ratio (MM footnote 6, p. 1616; COWY p. 98).
#
# The real-time combination (COWY section 4.3, pp. 106-107, Table 5 Panel A):
# at the start of each month t after an initial training period of K months,
# on an expanding window of all months before t,
#   1. c_t gives the managed and the original factor the same variance over the window;
#   2. the means mu and the covariance Sigma of the managed and the original factor
#      are estimated over the window;
#   3. the weights are (x_sigma, x) = (1 / gamma) Sigma^-1 mu;
#   4. the position in the original factor is y_t = x_sigma c_t / RV_{t-1} + x, with |y_t| <= cap;
#   5. the month's return is y_t f_t.
# S2, the real-time original, is the same with the original factor alone:
# y_t = (1 / gamma) mu / sigma^2, under the same cap. S3, the ex post optimal
# combination, uses the moments of the whole out-of-sample period.
REALTIME_K = 120               # months; COWY p. 106, "K = 120 months"
REALTIME_GAMMA = 5.0           # risk aversion; COWY Table 5 note, p. 106
REALTIME_LEVERAGE_CAP = 5.0    # |y_t| <= 5; COWY Table 5 note, p. 106, and p. 107
REALTIME_WINDOW = "expanding"  # COWY p. 106

# Scaled momentum (BSC equation 6, p. 115): WML scaled by sigma_target / sigma_hat_{t-1},
# sigma_hat the square root of equation 5's variance, sigma_target an annualised
# volatility of 12 per cent (BSC p. 115), here as a monthly figure.
BSC_TARGET_VOL_ANNUAL = 0.12
BSC_TARGET_VOL_MONTHLY = BSC_TARGET_VOL_ANNUAL / 12 ** 0.5

# ---------------------------------------------------------------------------
# 4. Measures and tests (the decisions with defaults of 8 November 2026)
# ---------------------------------------------------------------------------

MONTHS_PER_YEAR = 12
# The regression f_sigma_t = alpha + beta f_t + e_t, monthly. Alpha is reported
# in per cent a year (times 12); the appraisal ratio is alpha / sd(e) times the
# square root of 12 (MM footnote 11, p. 1620); Sharpe ratios are annualised by
# the square root of 12.
ALPHA_ANNUALISE = MONTHS_PER_YEAR
SHARPE_ANNUALISE = MONTHS_PER_YEAR ** 0.5

# Standard errors. E1 is judged with heteroskedasticity-robust standard errors,
# the papers' own (MM Table I note, p. 1619; COWY Table 3 note, White 1980,
# p. 104). The Newey and West standard error at 12 lags is reported beside it
# and is the one E4 to E6 use. HML's published alpha, 1.97 with a standard
# error of 1.02, has a t-statistic of 1.93 against the 1.96 line, so the
# choice can move a factor across it.
SE_E1 = "HC0"                  # White (1980)
NEWEY_WEST_LAGS = 12
SIGNIFICANCE_LEVEL = 0.05

# The Sharpe ratio difference (Ledoit and Wolf 2008): the studentised circular
# block bootstrap on paired series.
BOOTSTRAP_BLOCK_MONTHS = 12
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_LEVEL = 0.95
BOOTSTRAP_SEED = 20261108      # the date of the design, as a number

# The size check, run before the test touches a factor: 1,000 simulated pairs
# with equal Sharpe ratios and the sample's length, Student-t shocks with five
# degrees of freedom, a common component with the market's correlation between
# the pair, no difference in means.
SIZE_CHECK_PAIRS = 1000
SIZE_CHECK_T_DOF = 5
SIZE_CHECK_SEED = 20261110

# Costs: one proportional cost per unit of one-way turnover, the turnover of a
# volatility-managed strategy in a month being the absolute change in its
# position, |w_t - w_{t-1}|. Factor portfolios are traded through liquid
# large-cap long-short books, for which 10 basis points per unit of notional
# traded is the order the literature on implementation costs reports (MM Table
# IV, p. 1625, takes 10 basis points from Frazzini, Israel and Moskowitz); the
# one sensitivity doubles it.
COST_PER_TURNOVER = 0.0010
COST_SENSITIVITY = 0.0020

# ---------------------------------------------------------------------------
# 5. The specifications, counted before the code runs
# ---------------------------------------------------------------------------

# A specification is one pair of return series compared on one sample. The
# list is written out so that nothing can be added after a run without a dated
# disclosure. Cost levels are sensitivities of the same specifications and are
# not counted again.
_SAMPLES = {
    "MM": "first month to April 2015",
    "COWY": "first month to December 2016",
    "AFTER": "May 2015 to August 2026",
}
SPECIFICATIONS = (
    tuple(("E1", f, "MM", "volatility-managed", "original") for f in FACTORS)
    + tuple(("E2 (i)", f, "COWY", "volatility-managed", "original") for f in FACTORS)
    + tuple(("E2 (ii, iii)", f, "COWY", "real-time combination S1", "real-time original S2") for f in FACTORS)
    + tuple(("E2 (iv)", f, "COWY", "real-time combination S1", "ex post optimal combination S3") for f in FACTORS)
    + tuple(("E5", f, "AFTER", "real-time combination S1", "real-time original S2") for f in FACTORS)
    + tuple(("E5, no verdict", f, "AFTER", "volatility-managed, c to April 2015", "original") for f in FACTORS)
    + (("E3", "WML", "BSC 1927-05 to 2011-12", "scaled WML", "WML"),
       ("E3, extended", "WML", "BSC 1927-05 to 2026-08", "scaled WML", "WML"))
)
SPECIFICATION_COUNT = 38
SPECIFICATION_COUNT_DATE = "2026-11-10"

# ---------------------------------------------------------------------------
# 6. The expectations, written before the code
# ---------------------------------------------------------------------------

# Each is a strict pass or fail, reported in the same voice whether met or
# not; no level moves after a run. The levels the code reads are the named
# constants; the text and the reason are what each notebook quotes.

E1_MIN_FACTORS = 5
E1_ALPHA_WITHIN_SE = 1.0
E1_APPRAISAL_BAND = 0.05
E1_MARKET_SHARPE_BAND = 0.05

E2_SHARPE_BAND = 0.05
E2_SIGN_MIN_FACTORS = 5
E2_ONLY_SIGNIFICANT = ("Mom",)
E2_S1_S3_BAND = 0.05

E3_SHARPE_BAND = 0.05
E3_DRAWDOWN_BAND = 0.05        # five percentage points of drawdown

E4_SIZE_RANGE = (0.035, 0.065)
E4_MIN_FACTORS = 5

E5_MAX_FACTORS_ABOVE = 2

E6_COST = COST_PER_TURNOVER
E6_MIN_NW_T = 2.0
E6_MARKET_PERCENTILES = (50, 75, 90, 99)
E6_PERCENTILE_BAND = 0.15      # P50, P75 and P90
E6_P99_BAND = 1.0

EXPECTATIONS = {
    "E1": {
        "date": "2026-11-10",
        "text": (
            "On MM's sample (each factor's first month to April 2015), for at least five of the six factors: "
            "the alpha's sign and its 5 per cent significance category, with heteroskedasticity-robust (White) "
            "standard errors, match Table I (p. 1619); the alpha lies within one published standard error of the "
            "published alpha; and the appraisal ratio, computed from Table I's alpha and RMSE with footnote 11's "
            "annualisation (p. 1620), lies within 0.05 of the paper's figure, the market's 0.33 from the table, with "
            "the text's 0.34 (p. 1620) and 0.33 (p. 1612) noted beside it. For the market, the volatility-managed "
            "Sharpe ratio also lies within 0.05 of 0.52 (Table V, Panel A, p. 1626), and the market counts as "
            "matching only if it does. Every factor's comparisons are reported, whatever the count, with the "
            "computed N beside the published N."
        ),
        "reason": (
            "The replication criterion fixed on 6 September 2026: sign and significance, one standard error, and a "
            "band of 0.05. Its Sharpe ratio leg is the appraisal ratio for every factor because the paper prints no "
            "Sharpe ratio for any factor but the market. French revises history, so exact equality is not expected; "
            "five of six allows one factor to miss on a revised series, and the miss is reported, not chased."
        ),
    },
    "E2": {
        "date": "2026-11-10",
        "text": (
            "On COWY's sample (August 1926, January 1927 or August 1963 to December 2016), for each of the six "
            "factors: (i) the original and the volatility-managed Sharpe ratios lie within 0.05 of their Table 1 "
            "(p. 99); (ii) the real-time combination's Sharpe ratio S1 and the real-time original's S2 lie within "
            "0.05 of Table 5, Panel A (p. 106); (iii) S1 minus S2 has the published sign for at least five of the "
            "six, and the only difference significant at 5 per cent by the Jobson and Korkie test is momentum's, "
            "as published; (iv) S1 minus S3, the ex post optimal combination, lies within 0.05 of the published "
            "figure for all six. All twenty-four comparisons are reported, whatever the count."
        ),
        "reason": (
            "The critique's baseline is a real-time combination of the volatility-managed and the original factor "
            "(section 4.3, pp. 106-107): the scaling constant alone does not move a Sharpe ratio (MM footnote 6, "
            "p. 1616; COWY p. 98), and the weights of the combination are what an investor cannot know in advance."
        ),
    },
    "E3": {
        "date": "2026-11-10",
        "text": (
            "On May 1927 to December 2011, the Sharpe ratios of unscaled and scaled WML lie within 0.05 of 0.53 and "
            "0.97 (BSC Table 3, p. 116), and the maximum drawdowns lie within five percentage points of 96.69 and "
            "45.20 per cent (p. 112). The first scaled month is two months after the paper's 1927:03 because "
            "French's daily file of the ten momentum portfolios starts on 3 November 1926."
        ),
        "reason": (
            "The paper's two headline results, on its own momentum series (Appendix A, p. 119); the Sharpe band is "
            "the one used throughout the study."
        ),
    },
    "E4": {
        "date": "2026-11-08",
        "text": (
            "The size check rejects between 3.5 and 6.5 per cent of the 1,000 simulated null pairs at the 5 per cent "
            "level; and on MM's sample the bootstrap's verdict on the Sharpe ratio difference agrees in sign and in "
            "significance with the Newey and West t-statistic on the alpha for at least five of the six factors."
        ),
        "reason": (
            "A test that is wrong about its own size is not used; the band is two standard errors of a proportion "
            "of 0.05 over 1,000 draws. The alpha and the Sharpe ratio difference are two readings of the same gain "
            "and should agree."
        ),
    },
    "E5": {
        "date": "2026-11-10",
        "text": (
            "On the months after MM's sample, May 2015 to August 2026 (136 months), with the paper's c fixed on the "
            "data to April 2015 and the real-time combination carried forward from its expanding window: for each "
            "factor, the Sharpe ratio of the real-time combination against the real-time original S2, as in COWY "
            "Table 5, and of the volatility-managed factor against the original factor itself, each difference with "
            "its Ledoit and Wolf bootstrap interval. On at most two of the six factors the real-time combination's "
            "Sharpe ratio exceeds S2's by more than the interval allows, and the median across the six of the Sharpe "
            "difference lies within plus or minus the median of the six intervals' half-widths. The same two counts "
            "are reported for the volatility-managed factor, in no verdict."
        ),
        "reason": (
            "The study's own question, with the prior the critique gives: gains an investor could not have set in "
            "real time should show little in months neither paper could have known. If more than two factors clear "
            "the interval, the expectation fails and the finding is reported as such."
        ),
    },
    "E6": {
        "date": "2026-11-10",
        "text": (
            "At 10 basis points per unit of turnover, the market's volatility-managed alpha over MM's sample stays "
            "positive and more than two Newey and West standard errors from zero (the paper: 3.98 per cent a year at "
            "10 basis points, an average absolute change in weight of 0.73 and a break-even cost of 56 basis points, "
            "Table IV, p. 1625). On MM's sample, the market's P50, P75 and P90 of the leverage lie within 0.15 of "
            "0.93, 1.59 and 2.64, and its P99 within 1.0 of 6.39 (Table V, Panel A, p. 1626); on COWY's sample, each "
            "factor's P50 lies within 0.15 of their Table 1, Panel D (p. 99), and its P99 within 1.0 of theirs (6.47, "
            "5.07, 5.89, 8.64, 5.02, 4.56)."
        ),
        "reason": (
            "MM report that the market strategy's gain survives moderate costs (Table IV) and print the distribution "
            "of its leverage (Table V); COWY print the 1st, 50th and 99th percentiles for every factor. The "
            "expectation says whether both hold on this vintage."
        ),
    },
}

In [ ]:
%%writefile src/sq/french_loader.py
"""
Ken French data library: download, record, parse, assemble.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with one or more blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: eight digits
for daily data (19260701), six for monthly (192607), four for annual (1927).
Missing values are coded -99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a Period (day, month
or year), in decimal returns, with the missing codes replaced by NaN. It also
writes a provenance record for every file it downloads: the URL, the download
time, the SHA-256 of the zip, and the CRSP vintage line French prints at the
top of each CSV. A result is only as reproducible as the vintage it was
computed on, and French revises history, so the record is part of every output.

Two facts about the daily files matter for a monthly variance built from them.
They carry Saturday trading days until 24 May 1952, when the New York Stock
Exchange stopped trading on Saturdays. The number of trading days in a calendar
month therefore runs from 17 to 27 before June 1952 and from 15 to 23 from
1953 on (counted on vintage 202609). The variance definitions in constants.py
say how each paper treats the number of trading days.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = cut_to_sample_end(chosen["frame"].copy())
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def cut_to_sample_end(frame: pd.DataFrame, end: str | None = None) -> pd.DataFrame:
    """The frame up to the end of the sample (constants.SAMPLE_END): a monthly block to that
    month, a daily block to the last day of that month, an annual block to that year. A later vintage of French's
    files carries months after the end, which the study does not use; the vintage in the provenance record still says
    which revision of history the months before the end came from. With no end set, the frame is returned as it is."""
    end = C.SAMPLE_END if end is None else end
    if end is None or not isinstance(frame.index, pd.PeriodIndex) or len(frame) == 0:
        return frame
    end_m = pd.Period(end, "M")
    freq = frame.index.freqstr or ""
    if freq.startswith("M"):
        limit = end_m
    elif freq.startswith("D"):
        limit = end_m.asfreq("D", "end")
    else:
        limit = end_m.asfreq(frame.index.freq, "end")
    return frame.loc[:limit]


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# The study's series, assembled from the files above
# ---------------------------------------------------------------------------

def _suffix(freq: str) -> str:
    if freq not in ("monthly", "daily"):
        raise ValueError(f"freq is 'monthly' or 'daily', not {freq!r}")
    return "" if freq == "monthly" else "_daily"


def factors(freq: str = "monthly", cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """The six factors of constants.FACTORS and the risk-free rate, in decimal returns, cut at the sample's end.

    Each factor comes from the file constants.FACTOR_SOURCE names; the columns are aligned on the union of the
    dates, so a factor whose file starts later (momentum, RMW and CMA) is NaN before its first date.
    """
    sfx = _suffix(freq)
    cols = {}
    frames = {}
    for f in C.FACTORS + ("RF",):
        key = (C.FACTOR_SOURCE[f] if f != "RF" else C.RF_SOURCE) + sfx
        if key not in frames:
            frames[key] = load_monthly(key, 0, cache_dir)
        cols[f] = frames[key][f]
    out = pd.DataFrame(cols)
    out.attrs["provenance"] = [frames[k].attrs["provenance"] for k in sorted(frames)]
    return out


def wml(freq: str = "monthly", cache_dir: str | Path = "data/french") -> pd.Series:
    """WML: the top decile minus the bottom decile of the ten value-weighted portfolios formed on momentum,
    in decimal returns, cut at the sample's end (Barroso and Santa-Clara, Appendix A)."""
    key = "mom10" + _suffix(freq)
    frame = load_monthly(key, C.WML_BLOCK, cache_dir)
    if C.WML_BLOCK_TITLE_MUST_CONTAIN not in frame.attrs["title"]:
        raise ValueError(f"block {C.WML_BLOCK} of {key} is {frame.attrs['title']!r}, not the value-weighted returns")
    out = (frame[C.WML_LONG] - frame[C.WML_SHORT]).rename("WML")
    out.attrs["provenance"] = frame.attrs["provenance"]
    return out


def trading_days_per_month(daily: pd.DataFrame | pd.Series) -> pd.Series:
    """The number of daily rows in each calendar month, indexed by month."""
    idx = daily.index
    months = idx.asfreq("M") if isinstance(idx, pd.PeriodIndex) else pd.PeriodIndex(idx, freq="M")
    return pd.Series(1, index=months).groupby(level=0).sum().rename("trading days")

In [ ]:
%%writefile src/sq/managed.py
"""
The signal and the two constructions built on it.

The signal is last month's realised variance of a factor, computed from the
factor's daily returns. Each paper defines it in its own way, and each
comparison with a paper's printed numbers uses that paper's definition
(constants.RV_BY_COMPARISON):

  rv_mm     Moreira and Muir, equation 2, p. 1616: each day's return minus the
            month's mean daily return, squared and summed over the month.
  rv_cowy   Cederburg, O'Doherty, Wang and Yan, equation 4, p. 98: 22 divided
            by the month's number of trading days J, times the sum of squared
            daily returns.
  rv_bsc    Barroso and Santa-Clara, equation 5, p. 115: 21 times the mean
            squared daily return over the last 126 trading days up to the
            month's last trading day.

The volatility-managed factor (Moreira and Muir, equation 1, p. 1616) holds
c / RV_{t-1} units of the factor in month t, with c set so that the managed
and the original factor have the same standard deviation over the sample.

The real-time combination (Cederburg et al., section 4.3, pp. 106-107) mixes
the volatility-managed and the original factor with mean-variance weights
estimated each month from the months before it only. Its position in the
original factor is y_t = x_sigma c_t / RV_{t-1} + x, truncated at plus or
minus the cap (p. 107: "a leverage constraint of |y_t| <= 5").

All returns are monthly decimal excess returns indexed by a monthly Period;
daily returns are indexed by a daily Period.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C


# ---------------------------------------------------------------------------
# The signal
# ---------------------------------------------------------------------------

def _by_month(daily: pd.Series):
    daily = daily.dropna()
    return daily.groupby(daily.index.asfreq("M"))


def rv_mm(daily: pd.Series) -> pd.Series:
    """Moreira and Muir's realised variance of each calendar month (equation 2)."""
    return _by_month(daily).apply(lambda r: float(((r - r.mean()) ** 2).sum())).rename("rv_mm")


def rv_cowy(daily: pd.Series) -> pd.Series:
    """Cederburg et al.'s realised variance of each calendar month (equation 4): 22 / J times the sum of squares."""
    g = _by_month(daily)
    return (C.RV_COWY_DAYS * g.apply(lambda r: float((r ** 2).sum())) / g.size()).rename("rv_cowy")


def rv_bsc(daily: pd.Series) -> pd.Series:
    """Barroso and Santa-Clara's variance at each month's last trading day (equation 5): 21 times the mean
    squared daily return over the last 126 trading days. NaN until 126 trading days are available."""
    daily = daily.dropna()
    mean_sq = (daily ** 2).rolling(C.RV_BSC_TRADING_DAYS, min_periods=C.RV_BSC_TRADING_DAYS).mean()
    month_end = mean_sq.groupby(mean_sq.index.asfreq("M")).last()
    return (C.RV_BSC_DAYS_PER_MONTH * month_end).rename("rv_bsc")


def lagged(rv: pd.Series) -> pd.Series:
    """The variance of month t-1 indexed at month t, the value a position for month t can use."""
    out = rv.copy()
    out.index = out.index + 1
    return out.rename(f"{rv.name}_lag")


# ---------------------------------------------------------------------------
# The volatility-managed factor
# ---------------------------------------------------------------------------

def volatility_managed(f: pd.Series, rv: pd.Series, start: str, end: str, c: float | None = None):
    """The volatility-managed factor on the months start to end.

    Returns (frame, c). The frame has one row per month with the factor's return f, last month's variance
    rv_lag, the position weight = c / rv_lag and the managed return = weight * f. With c None, c is set so
    that the managed and the original return have the same standard deviation over these months (Moreira
    and Muir, p. 1616); a given c is used as it is, for months after the sample c was set on.
    """
    df = pd.DataFrame({"f": f, "rv_lag": lagged(rv)}).loc[start:end].dropna()
    raw = df["f"] / df["rv_lag"]
    if c is None:
        c = float(df["f"].std() / raw.std())
    df["weight"] = c / df["rv_lag"]
    df["managed"] = df["weight"] * df["f"]
    return df, c


# ---------------------------------------------------------------------------
# The real-time combination
# ---------------------------------------------------------------------------

def mean_variance_weights(returns: np.ndarray, gamma: float) -> np.ndarray:
    """(1 / gamma) Sigma^-1 mu for the columns of `returns` (months in rows), with the sample covariance."""
    mu = returns.mean(axis=0)
    sigma = np.atleast_2d(np.cov(returns, rowvar=False))
    return np.linalg.solve(sigma, mu) / gamma


def real_time(f: pd.Series, rv: pd.Series, first: str, end: str, k: int = C.REALTIME_K,
              gamma: float = C.REALTIME_GAMMA, cap: float = C.REALTIME_LEVERAGE_CAP) -> pd.DataFrame:
    """Cederburg et al.'s real-time combination (S1) and real-time original (S2), month by month.

    The months run from `first` to `end`; the first k are the initial training period and carry no
    position. For each later month t the window is every month from `first` to t-1 (an expanding window):

      1. c_t gives the managed and the original factor the same variance over the window;
      2. the means and the covariance of the managed and the original factor over the window give the
         weights (x_sigma, x) = (1 / gamma) Sigma^-1 mu;
      3. the position in the original factor is y_t = x_sigma c_t / RV_{t-1} + x, truncated at +-cap;
      4. S1's return is y_t f_t.

    S2 holds y2_t = (1 / gamma) mu_f / sigma_f^2 of the original factor alone, under the same cap, and
    returns y2_t f_t. Nothing from month t or later enters month t's positions.
    """
    df = pd.DataFrame({"f": f, "rv_lag": lagged(rv)}).loc[first:end].dropna()
    fr = df["f"].to_numpy()
    rv_lag = df["rv_lag"].to_numpy()
    raw = fr / rv_lag
    rows = []
    for i in range(k, len(df)):
        f_win, raw_win = fr[:i], raw[:i]
        c_t = f_win.std(ddof=1) / raw_win.std(ddof=1)
        x_sigma, x = mean_variance_weights(np.column_stack([c_t * raw_win, f_win]), gamma)
        y = x_sigma * c_t / rv_lag[i] + x
        y_capped = float(np.clip(y, -cap, cap))
        y2 = float(np.clip(f_win.mean() / (gamma * f_win.var(ddof=1)), -cap, cap))
        rows.append({
            "month": df.index[i], "window": i, "c": c_t, "x_sigma": x_sigma, "x": x,
            "y_uncapped": y, "y": y_capped, "at_cap": abs(y) > cap,
            "f": fr[i], "s1": y_capped * fr[i], "y2": y2, "s2": y2 * fr[i],
        })
    out = pd.DataFrame(rows).set_index("month")
    out.index = pd.PeriodIndex(out.index, freq="M")
    return out


def percentiles(s: pd.Series, levels=(5, 50, 95)) -> dict:
    """The given percentiles of a series and its maximum, keyed P5, P50, ..., max."""
    out = {f"P{p}": float(np.percentile(s.dropna(), p)) for p in levels}
    out["max"] = float(s.max())
    return out

In [ ]:
import sys
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
from sq import constants as C, french_loader as fl, managed as M

records = [fl.download(key) for key in C.FRENCH_FILES]
fl.write_provenance(records)
VINTAGE = sorted({r["vintage_line"].split("the ")[-1].split(" ")[0] for r in records})[-1]
print("vintage:", VINTAGE)

monthly = fl.factors("monthly")
daily = fl.factors("daily")
pd.set_option("display.width", 160)

## 2. The signal

The cell computes each definition for each factor from its daily returns. Each value belongs to the month whose trading days it was built from; the position for month t uses the value of month t-1, which `M.lagged` puts on month t.

In [ ]:
rv = {}
for f in C.FACTORS:
    d = daily[f].dropna()
    rv[f] = pd.DataFrame({"mm": M.rv_mm(d), "cowy": M.rv_cowy(d), "bsc": M.rv_bsc(d)})

counts = pd.DataFrame({f: {"months": len(rv[f]), "first": str(rv[f].index[0]), "last": str(rv[f].index[-1]),
                           "first Barroso-Santa-Clara value": str(rv[f]["bsc"].first_valid_index())} for f in C.FACTORS}).T
print(counts.to_string())

days = fl.trading_days_per_month(daily["Mkt-RF"].dropna())
ratio = rv["Mkt-RF"]["cowy"] / rv["Mkt-RF"]["mm"]
print()
print("market, Cederburg et al. over Moreira and Muir, median before June 1952:", round(ratio.loc[:"1952-05"].median(), 3),
      " (22 / (mean trading days - 1):", round(22 / (days.loc[:"1952-05"].mean() - 1), 3), ")")
print("market, Cederburg et al. over Moreira and Muir, median from 1953:      ", round(ratio.loc["1953-01":].median(), 3),
      " (22 / (mean trading days - 1):", round(22 / (days.loc["1953-01":].mean() - 1), 3), ")")
print("correlation of the logs of the two definitions, market:", round(np.log(rv["Mkt-RF"]["cowy"]).corr(np.log(rv["Mkt-RF"]["mm"])), 4))

flagged = {f: int((fl.trading_days_per_month(daily[f].dropna()) < C.TRADING_DAYS_FLAG_BELOW).sum()) for f in C.FACTORS}
print("months below", C.TRADING_DAYS_FLAG_BELOW, "trading days:", flagged)
assert all(v == 0 for v in flagged.values())

## 3. A worked month: October 1987

The market's daily returns in October 1987, the sum Moreira and Muir's definition takes, and the position the volatility-managed factor holds in November 1987. The constant c comes from Moreira and Muir's sample, section 4 of this notebook computes it for every factor.

In [ ]:
oct87 = daily["Mkt-RF"].loc["1987-10-01":"1987-10-31"]
by_hand = ((oct87 - oct87.mean()) ** 2).sum()
print(len(oct87), "trading days; largest one-day fall:", round(oct87.min() * 100, 2), "per cent on", oct87.idxmin())
print("largest one-day fall since 1926:", daily["Mkt-RF"].idxmin())
print("mean daily return:", round(oct87.mean() * 100, 3), "per cent")
print("realised variance, Moreira and Muir, by hand:", round(by_hand, 6), " from the module:", round(rv["Mkt-RF"].loc["1987-10", "mm"], 6))
assert abs(by_hand - rv["Mkt-RF"].loc["1987-10", "mm"]) < 1e-15

mkt_mm, c_mkt = M.volatility_managed(monthly["Mkt-RF"], rv["Mkt-RF"]["mm"], C.FACTOR_FIRST_MONTH["Mkt-RF"], C.MM_SAMPLE_END)
print("c on Moreira and Muir's sample:", round(c_mkt, 6))
print("position in November 1987: c / RV(October) =", round(c_mkt, 6), "/", round(by_hand, 6), "=", round(mkt_mm.loc["1987-11", "weight"], 3))
print("median position on the sample:", round(mkt_mm["weight"].median(), 3))
print("realised variance of October 1987 is the largest from 1980 to 1989:", rv["Mkt-RF"]["mm"].loc["1980-01":"1989-12"].idxmax() == pd.Period("1987-10", "M"))

## 4. The volatility-managed factor on Moreira and Muir's sample

For each factor, on its months from its first month to April 2015: c, and the 5th, 50th and 95th percentiles and the maximum of the position c / RV(t-1), with Moreira and Muir's definition of the signal.

In [ ]:
rows, vm = [], {}
for f in C.FACTORS:
    df, c = M.volatility_managed(monthly[f], rv[f]["mm"], C.FACTOR_FIRST_MONTH[f], C.MM_SAMPLE_END)
    vm[f] = df
    rows.append({"factor": f, "months": len(df), "c": c, **M.percentiles(df["weight"])})
vm_table = pd.DataFrame(rows).set_index("factor")
print(vm_table.round(4).to_string())
assert (vm_table["months"] == pd.Series(C.MM_EXPECTED_N)).all()
print("95th percentile over the median:", (vm_table["P95"] / vm_table["P50"]).round(2).to_dict())
for f in C.FACTORS:
    assert abs(vm[f]["managed"].std() - vm[f]["f"].std()) < 1e-12

## 5. The real-time combination

**One month worked in full.** The market's first month with a position, August 1936. The training window holds the 120 months from August 1926 to July 1936. The cell computes c for the window, the means and the covariance matrix of the managed and the original factor over the window, the mean-variance weights, and the position. It then repeats the weights with c multiplied by 7 to show that the position does not move.

In [ ]:
f_m, rv_m = monthly["Mkt-RF"], rv["Mkt-RF"]["cowy"]
window = pd.period_range(C.FACTOR_FIRST_MONTH["Mkt-RF"], "1936-07", freq="M")
f_w = f_m.loc[window].to_numpy()
raw_w = f_w / M.lagged(rv_m).loc[window].to_numpy()
c_t = f_w.std(ddof=1) / raw_w.std(ddof=1)
X = np.column_stack([c_t * raw_w, f_w])
print("window:", len(window), "months; c =", round(c_t, 6))
print("means (managed, original), per cent a month:", (X.mean(axis=0) * 100).round(3))
print("covariance matrix:"); print(np.cov(X, rowvar=False).round(6))
x_sigma, x = M.mean_variance_weights(X, C.REALTIME_GAMMA)
y = x_sigma * c_t / rv_m.loc["1936-07"] + x
print("weights: managed", round(x_sigma, 4), " original", round(x, 4))
print("position in August 1936: x_sigma * c / RV(July 1936) + x =", round(y, 4), "; after the cap:", round(float(np.clip(y, -5, 5)), 4))
xs7, x7 = M.mean_variance_weights(np.column_stack([7 * c_t * raw_w, f_w]), C.REALTIME_GAMMA)
print("with c times 7: managed weight", round(xs7, 4), "; x_sigma * c unchanged:", np.isclose(xs7 * 7 * c_t, x_sigma * c_t))
assert np.isclose(xs7 * 7 * c_t, x_sigma * c_t) and np.isclose(x7, x)

rt_mkt = M.real_time(f_m, rv_m, C.FACTOR_FIRST_MONTH["Mkt-RF"], C.SAMPLE_END)
assert np.isclose(rt_mkt.loc["1936-08", "y"], float(np.clip(y, -5, 5)))

**Every month, every factor.** The real-time combination and the real-time original from the end of the first training window to August 2026, with Cederburg et al.'s definition of the signal. The table gives the first month with a position, the number of months, the percentiles and the maximum of S1's position, the share of months in which it is negative (the combination is then short the factor, holding minus the factor's return), the share in which the cap binds, and the median position of S2.

In [ ]:
rows, rt = [], {}
for f in C.FACTORS:
    out = M.real_time(monthly[f], rv[f]["cowy"], C.FACTOR_FIRST_MONTH[f], C.SAMPLE_END)
    rt[f] = out
    rows.append({"factor": f, "first month": str(out.index[0]), "months": len(out), **M.percentiles(out["y"]),
                 "share below 0": (out["y"] < 0).mean(), "share at cap": out["at_cap"].mean(), "S2 median": out["y2"].median()})
rt_table = pd.DataFrame(rows).set_index("factor")
print(rt_table.round(4).to_string())
assert list(rt_table["first month"]) == ["1936-08"] * 3 + ["1937-01"] + ["1973-08"] * 2
assert list(rt_table["months"]) == [1081] * 3 + [1076] + [637] * 2

## 6. The market's signal and both positions

Three panels, one per series. The first shows the market's realised volatility, the square root of 12 times Moreira and Muir's realised variance, in per cent a year. The second shows the volatility-managed factor's position on Moreira and Muir's sample, on a logarithmic scale so that a position of 0.5 and one of 2 sit at the same distance from 1; the position is always above 0. The third shows the real-time combination's position from August 1936 on an ordinary scale, because it can be negative, with the cap at 5 and -5 as dashed lines.

In [ ]:
import matplotlib.pyplot as plt

NAVY, GREY, RULE, LIMIT = "#16213a", "#8d9ab0", "#d6d9de", "#5d6470"
vol = np.sqrt(12 * rv["Mkt-RF"]["mm"]) * 100
fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(10, 9), sharex=True, gridspec_kw={"hspace": 0.35})
fig.patch.set_facecolor("white")
a1.plot(vol.index.to_timestamp(), vol.values, color=NAVY, linewidth=0.8)
a1.set_title("Market realised volatility, % a year, July 1926 to August 2026", loc="left", fontsize=11, color=NAVY)
a2.plot(vm["Mkt-RF"].index.to_timestamp(), vm["Mkt-RF"]["weight"], color=NAVY, linewidth=0.8)
a2.set_yscale("log")
a2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
a2.axhline(1, color=RULE, linewidth=0.8)
a2.set_title("Volatility-managed position, multiple of the factor, log scale, August 1926 to April 2015", loc="left", fontsize=11, color=NAVY)
a3.plot(rt["Mkt-RF"].index.to_timestamp(), rt["Mkt-RF"]["y"], color=NAVY, linewidth=0.8)
a3.axhline(0, color=RULE, linewidth=0.8)
for level in (C.REALTIME_LEVERAGE_CAP, -C.REALTIME_LEVERAGE_CAP):
    a3.axhline(level, color=LIMIT, linewidth=1.0, linestyle="--")
a3.text(rt["Mkt-RF"].index[-1].to_timestamp(), C.REALTIME_LEVERAGE_CAP, " cap", va="bottom", fontsize=9, color=LIMIT)
a3.set_ylim(-5.8, 5.8)
a3.set_title("Real-time combination's position, multiple of the factor, August 1936 to August 2026", loc="left", fontsize=11, color=NAVY)
for a in (a1, a2, a3):
    a.set_facecolor("white")
    a.spines[["top", "right"]].set_visible(False)
    a.tick_params(labelsize=9)
plt.show()

## 7. The outputs

The signal of every factor under the three definitions, the volatility-managed positions on Moreira and Muir's sample, the real-time positions and returns to August 2026, and the two summary tables, each with the vintage in its file name.

In [ ]:
pd.concat(rv, axis=1).to_csv(f"outputs/realised_variance_{VINTAGE}.csv")
pd.concat({f: vm[f][["f", "rv_lag", "weight", "managed"]] for f in C.FACTORS}, axis=1).to_csv(f"outputs/volatility_managed_mm_sample_{VINTAGE}.csv")
pd.concat({f: rt[f] for f in C.FACTORS}, axis=1).to_csv(f"outputs/real_time_{VINTAGE}.csv")
vm_table.to_csv(f"outputs/position_volatility_managed_{VINTAGE}.csv")
rt_table.to_csv(f"outputs/position_real_time_{VINTAGE}.csv")
print(sorted(os.listdir("outputs")))

## What this notebook established, and what could be wrong

| Check | Expected | Where it comes from |
|---|---|---|
| Months below 15 trading days | none | notebook 01 |
| Market, Cederburg et al.'s signal over Moreira and Muir's, median | near 0.94 before June 1952, near 1.10 from 1953 | 22 / (J - 1): the rescaling to 22 days and the month's own mean |
| Volatility-managed position | median near 1, 95th percentile more than twice the median | c sets the managed factor's standard deviation equal to the original's; the published percentiles |
| October 1987 | the largest realised variance of the 1980s, a small November position | the fall of 19 October 1987 |
| Real-time combination, first month and months to August 2026 | August 1936 and 1,081 for Mkt-RF, SMB, HML; January 1937 and 1,076 for Mom; August 1973 and 637 for RMW, CMA | a training window of 120 months |
| The real-time position and c | unchanged when c is multiplied by 7 | the weights scale by the inverse of c |

Every assert in the notebook passed if the notebook ran to this cell. The percentiles of the two positions are reported with no pass or fail; E6 judges the volatility-managed position's percentiles in notebook 08.

What could be wrong:

- **Moreira and Muir's equation 2 is written for a 22-day month.** The module removes the month's own mean, the sum of its daily returns divided by its number of trading days. With the mean as small as it is next to the daily swings, the choice moves the signal by little; notebook 03 shows whether the computed table matches theirs.
- **The cap is applied by truncation.** Cederburg et al. state the constraint, |y_t| at most 5 (p. 107), without saying how a larger position is brought back to it; truncating at 5 is the direct reading. The share of months at the cap says how often the choice matters.
- **The vintage differs from the papers'.** French's history has been revised since 2015 and 2017. The provenance record says which vintage every number comes from.